# À travers les yeux d’une IA

*Atelier découverte de la vision par ordinateur — AI UpSkill*

Dans ce carnet, vous allez :

1. **appliquer des filtres à la main**, comme dans les diapos ;
2. **entraîner un modèle** et regarder les filtres qu’il invente seul ;
3. **regarder dans la tête d’un grand modèle**, entraîné sur plus d’un million de photos ;
4. **en équipe, construire votre propre modèle** et le mettre à l’épreuve.

### Mode d’emploi

- **Commencez par** *Fichier → Enregistrer une copie dans Drive*, sinon vos modifications seront perdues.
- Une cellule grise = un bloc de code. Cliquez dedans, puis **Maj + Entrée** pour l’exécuter.
- Exécutez les cellules **dans l’ordre**, de haut en bas.
- Les cellules marquées ✏️ sont faites pour être modifiées. Les autres, vous pouvez les lire, ou pas.
- Si tout se bloque : *Exécution → Redémarrer la session*, puis réexécutez depuis le début.

Pas besoin de carte graphique : tout tourne sur le processeur de Colab.

In [ ]:
# @title 🧰 Préparation : exécutez cette cellule sans la lire
import io
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from skimage import color, data

warnings.filterwarnings("ignore")
torch.manual_seed(0)
np.random.seed(0)
appareil = "cuda" if torch.cuda.is_available() else "cpu"

CATEGORIES = ["avion", "voiture", "oiseau", "chat", "cerf",
              "chien", "grenouille", "cheval", "bateau", "camion"]


def nombre(n):
    """45000 → « 45 000 »."""
    return f"{n:,}".replace(",", " ")


def pct(p, decimales=0):
    """0.634 → « 63 % », à la française."""
    return f"{100 * float(p):.{decimales}f}".replace(".", ",") + " %"


def montrer(images, titres=None, colonnes=None, taille=3, cmap="gray"):
    """Affiche des images côte à côte."""
    images = [np.asarray(i) for i in images]
    colonnes = colonnes or len(images)
    lignes = -(-len(images) // colonnes)
    fig, axes = plt.subplots(lignes, colonnes, figsize=(taille * colonnes, taille * lignes), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for i, (ax, img) in enumerate(zip(axes.flat, images)):
        ax.imshow(img, cmap=cmap if img.ndim == 2 else None, interpolation="nearest")
        if titres:
            ax.set_title(titres[i], fontsize=10)
    plt.tight_layout()
    plt.show()


def appliquer_filtre(image, filtre):
    """Fait glisser le filtre sur toute l’image. Renvoie la force de la réponse : 0 = rien."""
    img = torch.tensor(image, dtype=torch.float32)[None, None]
    k = torch.tensor(filtre, dtype=torch.float32)[None, None]
    return F.conv2d(img, k)[0, 0].abs().numpy()


def afficher_filtres(poids, titre=None, colonnes=16):
    """Chaque filtre en pixels : du plus négatif (sombre) au plus positif (clair)."""
    w = poids.detach().cpu().float()
    colonnes = min(colonnes, len(w))
    lignes = -(-len(w) // colonnes)
    fig, axes = plt.subplots(lignes, colonnes, figsize=(0.8 * colonnes, 0.8 * lignes + 0.5), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, f in zip(axes.flat, w):
        f = (f - f.min()) / (f.max() - f.min() + 1e-8)
        ax.imshow(f.permute(1, 2, 0).numpy() if len(f) == 3 else f[0].numpy(), cmap="gray", interpolation="nearest")
    if titre:
        fig.suptitle(titre)
    plt.tight_layout()
    plt.show()


# --- Données --------------------------------------------------------------

def telecharger_cifar10(partie):
    """CIFAR-10 depuis le miroir Hugging Face : les mêmes images, bien plus vite que le serveur d’origine."""
    from huggingface_hub import hf_hub_download
    import pandas as pd
    fichier = hf_hub_download("uoft-cs/cifar10", f"plain_text/{partie}-00000-of-00001.parquet", repo_type="dataset")
    tableau = pd.read_parquet(fichier)
    images = np.stack([np.array(Image.open(io.BytesIO(i["bytes"]))) for i in tableau["img"]])
    return torch.tensor(images).permute(0, 3, 1, 2), torch.tensor(tableau["label"].values)


# --- Entraînement --------------------------------------------------------

def preparer(x):
    """Images 0–255 → valeurs centrées autour de 0, ce que les modèles préfèrent."""
    return (x.float() / 255 - 0.5) / 0.25


def augmenter(x):
    """Variété gratuite : miroir gauche-droite pour la moitié des images, décalage de ±4 pixels."""
    miroir = torch.rand(len(x)) < 0.5
    x = torch.where(miroir[:, None, None, None], x.flip(3), x)
    dy, dx = np.random.randint(0, 9, size=2)
    return F.pad(x, (4, 4, 4, 4), mode="reflect")[:, :, dy:dy + 32, dx:dx + 32]


@torch.no_grad()
def predire(modele, x, taille_lot=1000):
    """Probabilités de chaque catégorie, pour chaque image."""
    modele.eval()
    return torch.cat([F.softmax(modele(preparer(x[i:i + taille_lot]).to(appareil)), 1).cpu()
                      for i in range(0, len(x), taille_lot)])


def precision(modele, x, y):
    return (predire(modele, x).argmax(1) == y).float().mean().item()


def entrainer(modele, x, y, validation, epoques=5, duree_max=120, taux=0.002, augmentation=False):
    """Montre les images au modèle, lot par lot, et corrige ses nombres après chaque lot."""
    modele.to(appareil)
    optimiseur = torch.optim.Adam(modele.parameters(), lr=taux)
    debut, prochain_message, vues = time.time(), 30, 0
    print(f"Entraînement sur {nombre(len(x))} images, {duree_max} secondes maximum…")
    for epoque in range(1, epoques + 1):
        modele.train()
        ordre = torch.randperm(len(x))
        for i in range(0, len(x), 128):
            lot = ordre[i:i + 128]
            images = preparer(x[lot])
            if augmentation:
                images = augmenter(images)
            perte = F.cross_entropy(modele(images.to(appareil)), y[lot].to(appareil))
            optimiseur.zero_grad()
            perte.backward()
            optimiseur.step()
            vues += len(lot)
            ecoule = time.time() - debut
            if ecoule > prochain_message:
                print(f"   … {ecoule:.0f} s, {nombre(vues)} images vues")
                prochain_message += 30
            if ecoule > duree_max:
                break
        p = precision(modele, *validation)
        print(f"Époque {epoque} : {pct(p)} de bonnes réponses sur des images jamais vues ({time.time() - debut:.0f} s)")
        if time.time() - debut > duree_max:
            print("⏱️ Temps écoulé !")
            break
    return modele


def montrer_predictions(modele, x, y, indices, colonnes=6):
    probas = predire(modele, x[indices])
    images, titres = [], []
    for p, i in zip(probas, indices):
        k = p.argmax().item()
        juste = k == y[i].item()
        images.append(x[i].permute(1, 2, 0).numpy())
        titres.append(f"{'✓' if juste else '✗'} {CATEGORIES[k]} ({pct(p[k])})"
                      + ("" if juste else f"\nc’était : {CATEGORIES[y[i]]}"))
    montrer(images, titres, colonnes=colonnes, taille=2.2)


print("✅ Prêt ! Calculs sur :", "carte graphique" if appareil == "cuda" else "processeur")

## 1. Les filtres à la main · 10 min

Pour un ordinateur, une image n’est qu’un tableau de nombres. Voici Chelsea, une chatte célèbre en vision par ordinateur : sa photo est fournie avec la bibliothèque *scikit-image*.

In [ ]:
chat = data.chelsea()            # une photo couleur : hauteur × largeur × 3 (rouge, vert, bleu)
gris = color.rgb2gray(chat)      # en niveaux de gris : 0 = noir, 1 = blanc

print("Taille de l’image :", gris.shape)
print("Un petit coin de 5 × 5 pixels, tel que l’ordinateur le voit :")
print(np.round(gris[100:105, 200:205], 2))
montrer([chat, gris], ["Chelsea", "En niveaux de gris"], taille=4)

Le filtre vertical des diapos : −1 à gauche, 0 au milieu, 1 à droite. On le fait glisser sur toute l’image ; à chaque position, on multiplie case par case et on additionne. Plus le résultat est clair, plus le filtre a réagi.

In [ ]:
vertical = [[-1, 0, 1],
            [-1, 0, 1],
            [-1, 0, 1]]

montrer([gris, appliquer_filtre(gris, vertical)], ["L’image", "Filtre vertical"], taille=4)

### 🤔 À vous de deviner

Que vont détecter ces deux filtres ? Pariez avant d’exécuter la cellule.

In [ ]:
horizontal = [[-1, -1, -1],
              [ 0,  0,  0],
              [ 1,  1,  1]]

diagonal = [[-1, -1, 0],
            [-1,  0, 1],
            [ 0,  1, 1]]

montrer([appliquer_filtre(gris, vertical), appliquer_filtre(gris, horizontal), appliquer_filtre(gris, diagonal)],
        ["Vertical", "Horizontal", "Diagonal"], taille=4)

### ✏️ Défi : inventez votre filtre

Changez les nombres ci-dessous, puis réexécutez. Quelques idées :

- tout à `1/9` : que se passe-t-il ?
- un `8` au centre, entouré de `-1` ;
- un filtre plus grand, 5 × 5 ou 7 × 7 : le code accepte toutes les tailles.

In [ ]:
mon_filtre = [[0, 0, 0],
              [0, 1, 0],
              [0, 0, 0]]   # ce filtre-là ne change rien : à vous de jouer !

montrer([gris, appliquer_filtre(gris, mon_filtre)], ["L’image", "Mon filtre"], taille=4)

## 2. Entraîner un modèle · 15 min

Jusqu’ici, c’est nous qui avons choisi les nombres des filtres. Maintenant, un modèle va les **trouver seul**.

Ses images d’entraînement : **CIFAR-10**, 60 000 petites photos de 32 × 32 pixels réparties en 10 catégories. On lui montre les photos avec la bonne réponse, et il corrige ses filtres petit à petit.

In [ ]:
# Téléchargement : environ 140 Mo, une minute environ.
x_tout, y_tout = telecharger_cifar10("train")
x_entr, y_entr = x_tout[:45000], y_tout[:45000]   # pour apprendre
x_val, y_val = x_tout[45000:], y_tout[45000:]     # pour vérifier pendant l’entraînement
x_test, y_test = telecharger_cifar10("test")      # le test final, à ne regarder qu’à la fin

print(f"{nombre(len(x_entr))} images pour apprendre, {nombre(len(x_val))} pour vérifier, {nombre(len(x_test))} pour le test final")
montrer([x_entr[i].permute(1, 2, 0) for i in range(18)], [CATEGORIES[y_entr[i]] for i in range(18)], colonnes=9, taille=1.6)

Voici le modèle : **trois couches de filtres**, puis un vote entre les 10 catégories. La première couche a 16 filtres de 5 × 5 pixels, et chacun regarde les 3 couleurs à la fois.

Au départ, leurs nombres sont tirés au hasard, comme dans les diapos.

In [ ]:
torch.manual_seed(0)
modele = nn.Sequential(
    nn.Conv2d(3, 16, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),    # couche 1 : 16 filtres de 5 × 5
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # couche 2 : 32 filtres
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # couche 3 : 64 filtres
    nn.Flatten(), nn.Linear(64 * 4 * 4, 10),                       # le vote : un score par catégorie
)
with torch.no_grad():
    modele[0].weight.mul_(0.1)   # de petits nombres au départ : ce qui sera appris ressortira mieux

filtres_avant = modele[0].weight.detach().clone()
afficher_filtres(filtres_avant, "Couche 1, avant l’entraînement : du bruit")

On lance l’entraînement : **2 minutes maximum**. Pendant ce temps, le modèle regarde des milliers d’images et ajuste ses nombres après chaque lot de 128.

Au hasard, il aurait 10 % de bonnes réponses. Voyons s’il fait mieux.

In [ ]:
entrainer(modele, x_entr, y_entr, validation=(x_val, y_val), epoques=4, duree_max=120)

Et ses filtres maintenant ? Comparez-les à ceux d’avant : personne ne les a dessinés.

Avec des images de 32 × 32 pixels et 2 minutes d’entraînement, ils restent flous : cherchez les côtés clairs et sombres (des bords) et les taches de couleur. Ceux d’un grand modèle, dans la partie 3, sont bien plus nets.

In [ ]:
afficher_filtres(filtres_avant, "Avant l’entraînement")
afficher_filtres(modele[0].weight, "Après l’entraînement")

### Ce que voit la couche 1

Faisons passer une image de chat dans la couche 1. Chaque filtre produit sa propre carte : certains réagissent aux bords, d’autres à certaines couleurs.

In [ ]:
i = (y_test == 3).nonzero()[0].item()   # la première image de chat du test
with torch.no_grad():
    cartes = modele[:2](preparer(x_test[i:i + 1]).to(appareil))[0].cpu()

montrer([x_test[i].permute(1, 2, 0)] + list(cartes),
        ["L’image"] + [f"Filtre {k + 1}" for k in range(len(cartes))], colonnes=9, taille=1.6)

### Le verdict

Des images que le modèle n’a jamais vues, avec sa réponse et sa confiance.

In [ ]:
montrer_predictions(modele, x_test, y_test, list(range(12)))
print("Bonnes réponses sur les 10 000 images de test :", pct(precision(modele, x_test, y_test)))

## 3. Dans la tête d’un grand modèle · 10 min

Notre petit modèle a appris pendant 2 minutes. **ResNet-50** a appris sur **1,2 million de photos** (1 000 catégories), pendant des jours, sur des cartes graphiques. Pas question de l’entraîner ici : on télécharge le résultat (environ 100 Mo) et on regarde dedans.

In [ ]:
from torchvision.models import ResNet50_Weights, resnet50

poids_resnet = ResNet50_Weights.IMAGENET1K_V2
resnet = resnet50(weights=poids_resnet).eval()
transfo = poids_resnet.transforms()
NOMS_IMAGENET = poids_resnet.meta["categories"]   # en anglais

def top_resnet(image, n=5):
    with torch.no_grad():
        p = F.softmax(resnet(transfo(image)[None]), 1)[0]
    meilleurs = p.topk(n)
    return [(NOMS_IMAGENET[k], v.item()) for v, k in zip(meilleurs.values, meilleurs.indices)]

chelsea = Image.fromarray(data.chelsea())
for nom, p in top_resnet(chelsea):
    print(f"{pct(p):>6}  {nom}")

Il reconnaît un chat, et même la race. Regardons ses **64 filtres de la première couche**, de 7 × 7 pixels : reconnaissez-vous ceux des diapos ?

In [ ]:
afficher_filtres(resnet.conv1.weight, "ResNet-50, couche 1")

### Chelsea, de couche en couche

ResNet-50 empile une cinquantaine de couches, regroupées en 5 niveaux. Pour chaque niveau, voici quelques cartes parmi les plus actives. Au début, on reconnaît encore la photo ; plus on avance, plus les cartes sont petites et abstraites. Elles ne disent plus « à quoi ça ressemble », mais « où se trouve ce que je cherche ».

In [ ]:
BLOCS = list(resnet.children())
FIN_NIVEAU = {1: 3, 2: 5, 3: 6, 4: 7, 5: 8}   # niveau → nombre de blocs à traverser

with torch.no_grad():
    x = transfo(chelsea)[None]
    for niveau in range(1, 6):
        for bloc in BLOCS[FIN_NIVEAU.get(niveau - 1, 0):FIN_NIVEAU[niveau]]:   # on avance d’un niveau
            x = bloc(x)
        cartes = x[0]
        actives = cartes.mean(dim=(1, 2)).topk(8).indices
        montrer([cartes[k] for k in actives],
                [f"Niveau {niveau} · {cartes.shape[1]} × {cartes.shape[2]}"] + [""] * 7,
                cmap="magma", taille=1.6)

### Les rêves du réseau

On part d’une image de bruit et on la modifie petit à petit pour **allumer au maximum un filtre précis**. L’image obtenue montre ce que ce filtre cherche.

Cette cellule calcule pendant deux à trois minutes.

In [ ]:
MOYENNE_IMAGENET = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
ECART_IMAGENET = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
# Dans les vraies photos, les couleurs sont liées : on les mélange comme dans la nature.
COULEURS = torch.tensor([[0.26, 0.09, 0.02], [0.27, 0.00, -0.05], [0.27, -0.09, 0.03]])
COULEURS = COULEURS / COULEURS.norm(dim=0).max()

def reves(niveau, canaux, iterations=96, taille=128):
    """Fabrique, pour chaque canal, l’image qui l’allume le plus."""
    debut_du_reseau = nn.Sequential(*BLOCS[:FIN_NIVEAU[niveau]]).eval()
    for p in debut_du_reseau.parameters():
        p.requires_grad_(False)
    # L’image est décrite par ses fréquences : ça évite un bruit de pixels illisible.
    fy = torch.fft.fftfreq(taille)[:, None]
    fx = torch.fft.rfftfreq(taille)[None, :]
    echelle = 1 / torch.sqrt(fx ** 2 + fy ** 2).clamp(min=1 / taille)
    spectre = (0.01 * torch.randn(len(canaux), 3, taille, taille // 2 + 1, 2)).requires_grad_()
    optimiseur = torch.optim.Adam([spectre], lr=0.05)

    def image():
        s = torch.view_as_complex(spectre) * echelle
        x = torch.fft.irfft2(s, s=(taille, taille), norm="ortho") / 4
        return torch.sigmoid(torch.einsum("nchw,dc->ndhw", x, COULEURS))

    for _ in range(iterations):
        # On bouge et on zoome un peu l’image à chaque fois : le motif doit marcher partout.
        x = F.pad(image(), (8, 8, 8, 8), mode="reflect")
        dy, dx = np.random.randint(0, 17, size=2)
        x = x[:, :, dy:dy + taille, dx:dx + taille]
        x = F.interpolate(x, scale_factor=float(np.random.choice([0.95, 1.0, 1.05])), mode="bilinear")
        activations = debut_du_reseau((x - MOYENNE_IMAGENET) / ECART_IMAGENET)
        perte = -activations[torch.arange(len(canaux)), canaux].mean()
        optimiseur.zero_grad()
        perte.backward()
        optimiseur.step()
    return list(image().detach().permute(0, 2, 3, 1).numpy())

for niveau, canaux, nom in [(2, [0, 3, 8, 13, 20, 41], "des contours et des rayures"),
                            (3, [1, 5, 18, 34, 60, 97], "des textures"),
                            (4, [3, 11, 28, 55, 140, 301], "des motifs plus complexes")]:
    t = time.time()
    montrer(reves(niveau, canaux), [f"Niveau {niveau}"] + [""] * 5, taille=2)
    print(f"Niveau {niveau} : {nom} ({time.time() - t:.0f} s)")

Contours, puis textures, puis des motifs de plus en plus complexes : c’est l’empilement des couches des diapos. Avec beaucoup plus de calcul et quelques astuces, on obtient les images de l’article [*Feature Visualization*](https://distill.pub/2017/feature-visualization/).

✏️ Changez les numéros de `canaux` (de 0 à 255 au niveau 2, 511 au niveau 3, 1 023 au niveau 4) pour explorer d’autres filtres.

## 4. En équipe : votre modèle · 20 min

### Règles du jeu

- Réglez votre modèle avec le formulaire ci-dessous, puis exécutez les deux cellules.
- **3 minutes d’entraînement maximum** : le chronomètre est dans le code.
- Le score est calculé sur les **10 000 images de test**, que le modèle n’a jamais vues.
- Annoncez votre score : l’animateur tient le classement. Vous pouvez réessayer autant que vous voulez.

### Les réglages

| Réglage | Ce qu’il change |
|---|---|
| `couches` | Plus de couches = des motifs plus complexes, mais un modèle plus lent. |
| `filtres` | Le nombre de filtres de la couche 1 (il double à chaque couche suivante). |
| `taille_des_filtres` | 3 × 3, 5 × 5 ou 7 × 7 pixels. |
| `images_d_entrainement` | Plus d’images = un meilleur apprentissage, mais chaque passage prend plus de temps. |
| `epoques` | Combien de fois le modèle voit toutes les images. |
| `augmentation` | Retourne et décale les images au hasard : le modèle voit plus de variété. |
| `taux_d_apprentissage` | La taille de chaque correction. Trop petit, il apprend lentement ; trop grand, il s’emballe. |

Tout est un compromis avec le chronomètre : à vous de trouver le meilleur.

In [ ]:
# @title ✏️ Réglages de votre modèle
nom_equipe = "Les Chats Pottés"  # @param {type:"string"}
couches = 3  # @param {type:"slider", min:1, max:4, step:1}
filtres = 16  # @param [8, 16, 32, 64] {type:"raw"}
taille_des_filtres = 3  # @param [3, 5, 7] {type:"raw"}
images_d_entrainement = 20000  # @param [5000, 10000, 20000, 45000] {type:"raw"}
epoques = 5  # @param {type:"slider", min:1, max:20, step:1}
augmentation = False  # @param {type:"boolean"}
taux_d_apprentissage = 0.002  # @param [0.0003, 0.001, 0.002, 0.005, 0.02] {type:"raw"}

In [ ]:
def construire_modele(couches, filtres, taille_des_filtres):
    etapes, entree, cote = [], 3, 32
    for c in range(couches):
        sortie = min(filtres * 2 ** c, 256)
        etapes += [nn.Conv2d(entree, sortie, taille_des_filtres, padding=taille_des_filtres // 2),
                   nn.BatchNorm2d(sortie), nn.ReLU(), nn.MaxPool2d(2)]
        entree, cote = sortie, cote // 2
    etapes += [nn.Flatten(), nn.Dropout(0.3), nn.Linear(entree * cote * cote, 10)]
    modele = nn.Sequential(*etapes)
    with torch.no_grad():
        modele[0].weight.mul_(0.1)   # de petits nombres au départ, comme dans la partie 2
    return modele

torch.manual_seed(0)
modele_equipe = construire_modele(couches, filtres, taille_des_filtres)
print(f"Votre modèle a {nombre(sum(p.numel() for p in modele_equipe.parameters()))} nombres à apprendre.\n")

entrainer(modele_equipe, x_entr[:images_d_entrainement], y_entr[:images_d_entrainement],
          validation=(x_val, y_val), epoques=epoques, duree_max=180,
          taux=taux_d_apprentissage, augmentation=augmentation)

score = precision(modele_equipe, x_test, y_test)
print(f"\n🏆 {nom_equipe} : {pct(score, 1)} de bonnes réponses sur 10 000 images jamais vues")

### Mettez-le à l’épreuve

Où votre modèle se trompe-t-il ?

In [ ]:
predictions = predire(modele_equipe, x_test)
choix = predictions.argmax(1)

par_categorie = [(choix[y_test == k] == k).float().mean().item() for k in range(10)]
plt.figure(figsize=(7, 3.5))
plt.barh(CATEGORIES, par_categorie, color="#f26522")
plt.xlim(0, 1)
plt.title("Bonnes réponses par catégorie")
plt.gca().invert_yaxis()
plt.show()

confusions = torch.zeros(10, 10, dtype=torch.long)
confusions.index_put_((y_test, choix), torch.ones_like(y_test), accumulate=True)
confusions.fill_diagonal_(0)
print("Ses confusions les plus fréquentes :")
for n in confusions.flatten().topk(5).indices:
    vrai, pris = divmod(n.item(), 10)
    print(f"   {CATEGORIES[vrai]} pris pour {CATEGORIES[pris]} : {confusions[vrai, pris].item()} fois")

Ses erreurs les plus **sûres d’elles** : les images où il s’est trompé avec le plus d’assurance. Vous comprenez pourquoi ?

In [ ]:
confiance = predictions.max(1).values
fausses = (choix != y_test).nonzero().flatten()
pires = fausses[confiance[fausses].argsort(descending=True)[:12]]
montrer_predictions(modele_equipe, x_test, y_test, pires.tolist())

Les filtres que **votre** modèle a inventés dans sa première couche :

In [ ]:
afficher_filtres(modele_equipe[0].weight, f"Couche 1 de « {nom_equipe} »")

### Votre propre image

Envoyez une photo (un animal, un véhicule…) et comparez votre modèle à ResNet-50. Sans photo, la cellule prend Chelsea.

Attention : votre modèle ne connaît que ses 10 catégories, et ne voit qu’en 32 × 32 pixels.

In [ ]:
try:
    from google.colab import files
    envoi = files.upload()
    image = Image.open(io.BytesIO(next(iter(envoi.values())))).convert("RGB")
except Exception:
    image = Image.fromarray(data.chelsea())

cote = min(image.size)
carree = image.crop(((image.width - cote) // 2, (image.height - cote) // 2,
                     (image.width + cote) // 2, (image.height + cote) // 2))
petite = carree.resize((32, 32), Image.LANCZOS)
montrer([carree, petite], ["Ce que vous voyez", "Ce que voit votre modèle"], taille=3.5)

p = predire(modele_equipe, torch.tensor(np.array(petite)).permute(2, 0, 1)[None])[0]
print(f"Votre modèle ({nom_equipe}) :")
for v, k in zip(*p.topk(3)):
    print(f"   {pct(v):>6}  {CATEGORIES[k]}")
print("ResNet-50 :")
for nom, v in top_resnet(carree, 3):
    print(f"   {pct(v):>6}  {nom}")

## Pour aller plus loin

- [**CNN Explainer**](https://poloclub.github.io/cnn-explainer/) : un réseau de vision qu’on peut explorer dans le navigateur, couche par couche.
- [**Feature Visualization**](https://distill.pub/2017/feature-visualization/) (Distill, 2017) : les rêves des réseaux, en très haute qualité.
- [**Tutoriels PyTorch**](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html) : entraîner un classifieur sur CIFAR-10, pas à pas.

Merci d’avoir participé ! 🐱